In [1]:
import pandas as pd

In [2]:
df=pd.read_csv("Meghana_Feature_Engineered.csv")

In [3]:
df[[
    "Avg_Rating",
    "Complaints",
    "Delivery_Time_Min",
    "Service_Quality_Score",
    "Operational_Risk_Score"
]].head()

,Avg_Rating,Complaints,Delivery_Time_Min,Service_Quality_Score,Operational_Risk_Score
0,3.7,228,43,67.22,39.333863
1,3.8,187,43,69.18,39.234105
2,4.0,197,47,72.04,43.039774
3,3.8,242,38,67.68,34.241552
4,3.9,226,36,70.92,32.135373


In [4]:
df['Operational_Risk_Score'].describe()

count    312.000000
mean      27.036639
std        8.684776
min       13.202426
25%       19.305133
50%       26.709176
75%       34.239378
max       43.235829
Name: Operational_Risk_Score, dtype: float64

# STep 1 : Create a Target Column

In [5]:
def assign_operational_risk(score):

    if score <= 19.305133:
        return "Low"

    elif score <= 34.239378:
        return "Medium"

    else:
        return "High"


df["Operational_Risk"] = (
    df["Operational_Risk_Score"]
    .apply(assign_operational_risk)
)

In [8]:
df.head()

,Branch_ID,Branch_Name,Region,Month,Orders,Revenue,Avg_Rating,Customer_Count,Repeat_Customers,New_Customers,...,Complaint_Rate_Pct,New_Customer_Rate,New_Customer_Rate_Pct,Service_Quality_Score,Customer_Acquisition_Score,Complaint_Per_1000_Orders,Revenue_Growth_Pct,Repeat_Customer_Growth_Pct,Churn_Risk,Operational_Risk
0,E_01,Electronic City,South,2024-01-01,6733,2316152,3.7,5639,2706,2933,...,3.39,0.5201,52.01,67.22,52.01,33.863063,0.000000,0.000000,High,High
1,E_02,Electronic City,South,2024-02-01,5483,2121921,3.8,4382,1928,2454,...,3.41,0.5600,56.00,69.18,56.00,34.105417,-8.385935,-28.750924,High,High
2,E_03,Electronic City,South,2024-03-01,4953,2060448,4.0,4240,1865,2375,...,3.98,0.5601,56.01,72.04,56.01,39.773874,-2.897045,-3.267635,Medium,High
3,E_04,Electronic City,South,2024-04-01,5824,2364544,3.8,4517,2213,2304,...,4.16,0.5101,51.01,67.68,51.01,41.552198,14.758732,18.659517,High,High
4,E_05,Electronic City,South,2024-05-01,6389,2287262,3.9,5487,2469,3018,...,3.54,0.5500,55.00,70.92,55.00,35.373298,-3.268368,11.568007,High,Medium


# Step 2 : Check whether Target Variable Balanced or not?

In [6]:
df['Operational_Risk'].value_counts()

Operational_Risk
Medium    156
High       78
Low        78
Name: count, dtype: int64

# Step 3: Select Features For Operational Risk Model 

In [7]:
X = df[
[
    "Avg_Rating",
    "Complaints",
    "Delivery_Time_Min",
    "Service_Quality_Score",
    "Complaint_Rate_Pct",
    "Customer_Loyalty_Score",
    "Repeat_Customer_Rate_Pct"
]
]

In [8]:
y = df["Operational_Risk"]

# Step 4: Label Encoding

In [9]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y = le.fit_transform(
    df["Operational_Risk"]
)

print(
    dict(
        zip(
            le.classes_,
            le.transform(le.classes_)
        )
    )
)

{'High': np.int64(0), 'Low': np.int64(1), 'Medium': np.int64(2)}


# STep 5: Train & Test Data 

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y
)

# Step 6: Verify Distribution 

In [11]:
import pandas as pd

print(
    pd.Series(y_train).value_counts(normalize=True)
)

print(
    pd.Series(y_test).value_counts(normalize=True)
)

2    0.502008
1    0.248996
0    0.248996
Name: proportion, dtype: float64
2    0.492063
0    0.253968
1    0.253968
Name: proportion, dtype: float64


# Step 7: Train Random Forest 

In [12]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

model.fit(
    X_train,
    y_train
)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

# Step 8: Prediction

In [13]:
y_pred = model.predict(
    X_test
)

# STep 9: Evaluation

In [14]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

accuracy = accuracy_score(
    y_test,
    y_pred
)

print(
    "Accuracy:",
    round(
        accuracy*100,
        2
    ),
    "%"
)

Accuracy: 98.41 %


In [15]:
print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        16
           1       0.94      1.00      0.97        16
           2       1.00      0.97      0.98        31

    accuracy                           0.98        63
   macro avg       0.98      0.99      0.98        63
weighted avg       0.99      0.98      0.98        63



In [16]:
print(
    confusion_matrix(
        y_test,
        y_pred
    )
)

[[16  0  0]
 [ 0 16  0]
 [ 0  1 30]]


# Step 9: Feature Importance 

In [19]:
importance_df = pd.DataFrame({

    "Feature": X.columns,

    "Importance":
    model.feature_importances_

})

importance_df = (
    importance_df
    .sort_values(
        "Importance",
        ascending=False
    )
)

importance_df

,Feature,Importance
2,Delivery_Time_Min,0.395081
5,Customer_Loyalty_Score,0.162098
6,Repeat_Customer_Rate_Pct,0.139190
3,Service_Quality_Score,0.109466
0,Avg_Rating,0.081760
4,Complaint_Rate_Pct,0.062141
1,Complaints,0.050264


In [18]:
def operational_risk_summary_table(df):

    risk_summary = (
        df.groupby("Branch_Name")
        .agg(
            Avg_Rating=("Avg_Rating", "mean"),
            Avg_Complaints=("Complaints", "mean"),
            Avg_Delivery_Time=("Delivery_Time_Min", "mean"),
            Avg_Service_Quality=("Service_Quality_Score", "mean"),
            Avg_Operational_Risk_Score=("Operational_Risk_Score", "mean"),
            Latest_Operational_Risk=("Operational_Risk", "last")
        )
        .reset_index()
    )

    risk_summary["Avg_Rating"] = risk_summary["Avg_Rating"].round(2)
    risk_summary["Avg_Complaints"] = risk_summary["Avg_Complaints"].round(0).astype(int)
    risk_summary["Avg_Delivery_Time"] = risk_summary["Avg_Delivery_Time"].round(1)
    risk_summary["Avg_Service_Quality"] = risk_summary["Avg_Service_Quality"].round(2)
    risk_summary["Avg_Operational_Risk_Score"] = risk_summary["Avg_Operational_Risk_Score"].round(2)

    def operational_reason(row):
        reasons = []

        if row["Avg_Delivery_Time"] > df["Delivery_Time_Min"].mean():
            reasons.append("High delivery time")

        if row["Avg_Complaints"] > df["Complaints"].mean():
            reasons.append("High complaints")

        if row["Avg_Service_Quality"] < df["Service_Quality_Score"].mean():
            reasons.append("Low service quality")

        if row["Avg_Rating"] < df["Avg_Rating"].mean():
            reasons.append("Below-average rating")

        if len(reasons) == 0:
            return "Operations are stable"

        return ", ".join(reasons)

    def operational_action(risk):
        if risk == "High":
            return "Fix delivery delays, reduce complaints, add peak-hour staff, monitor service quality daily"
        elif risk == "Medium":
            return "Monitor weekly, improve delivery coordination, train staff, reduce complaint causes"
        else:
            return "Maintain standards and use this branch as an operational benchmark"

    risk_summary["Main_Reason"] = risk_summary.apply(operational_reason, axis=1)
    risk_summary["Recommended_Action"] = risk_summary["Latest_Operational_Risk"].apply(operational_action)

    risk_summary = risk_summary[
        [
            "Branch_Name",
            "Latest_Operational_Risk",
            "Avg_Operational_Risk_Score",
            "Avg_Rating",
            "Avg_Complaints",
            "Avg_Delivery_Time",
            "Avg_Service_Quality",
            "Main_Reason",
            "Recommended_Action"
        ]
    ]

    return risk_summary


In [19]:
operational_table = operational_risk_summary_table(df)
operational_table

,Branch_Name,Latest_Operational_Risk,Avg_Operational_Risk_Score,Avg_Rating,Avg_Complaints,Avg_Delivery_Time,Avg_Service_Quality,Main_Reason,Recommended_Action
0,Electronic City,High,37.76,3.85,207,41.6,69.98,"High delivery time, High complaints, Low servi...","Fix delivery delays, reduce complaints, add pe..."
1,Indiranagar,Low,16.27,4.65,58,20.9,92.09,Operations are stable,Maintain standards and use this branch as an o...
2,Jayanagar,Medium,17.14,4.65,61,21.8,92.22,Operations are stable,"Monitor weekly, improve delivery coordination,..."
3,Kalyan Nagar,High,37.14,3.85,194,41.0,70.28,"High delivery time, High complaints, Low servi...","Fix delivery delays, reduce complaints, add pe..."
4,Kanakapura Road,Medium,26.90,4.28,113,31.2,83.17,Operations are stable,"Monitor weekly, improve delivery coordination,..."
5,Koramangala,Low,17.45,4.68,67,22.1,92.73,Operations are stable,Maintain standards and use this branch as an o...
6,Mahadevapura,High,39.08,3.87,203,42.9,70.61,"High delivery time, High complaints, Low servi...","Fix delivery delays, reduce complaints, add pe..."
7,Marathahalli,Medium,26.14,4.29,111,30.4,83.51,Operations are stable,"Monitor weekly, improve delivery coordination,..."
8,Rajarajeshwari Nagar,High,37.63,3.82,206,41.4,69.39,"High delivery time, High complaints, Low servi...","Fix delivery delays, reduce complaints, add pe..."
9,Residency Road,Low,16.61,4.65,63,21.2,92.00,Operations are stable,Maintain standards and use this branch as an o...


# Most Important Discovery

## Delivery Time = 39.5%

Almost:

```text
40%
```

of operational risk is explained by delivery performance.

This is HUGE.

---

### What CEO Learns

Most managers think:

```text
Complaints cause operational problems.
```

Model says:

```text
NO.

Delivery delays are the biggest problem.
```

Because:

```text
Slow Delivery
↓
Customer Frustration
↓
Complaints
↓
Lower Ratings
↓
Lower Loyalty
↓
Business Problems
```

Delivery delay is the root cause.

---

# Second Biggest Driver

## Customer Loyalty Score

```text
16.2%
```

Meaning:

```text
Branches with weak loyalty
are more likely to face
operational problems.
```

---

# Third Biggest Driver

## Repeat Customer Rate

```text
13.9%
```

Meaning:

```text
Customers voting with their wallet.
```

If customers stop returning:

```text
Something operational is wrong.
```

---

# VERY INTERESTING

Look here:

```text
Complaints = 5%

Complaint Rate = 6%
```

Everyone assumes complaints are the biggest issue.

Model says:

```text
Complaints are actually
a symptom.

Not the root cause.
```

ROOT CAUSE:

```text
Delivery Time
```

This is exactly the kind of insight CEOs pay consultants lakhs for.



# STep 10: Testing with Scenario Data 

# Scenario 1 — Healthy Branch

In [20]:
healthy_branch = pd.DataFrame({

    "Avg_Rating":[4.8],

    "Complaints":[20],

    "Delivery_Time_Min":[18],

    "Service_Quality_Score":[95],

    "Complaint_Rate_Pct":[0.3],

    "Customer_Loyalty_Score":[82],

    "Repeat_Customer_Rate_Pct":[82]

})

In [ ]:
# Scenario 1 — Moderate Branch

In [21]:
medium_branch = pd.DataFrame({

    "Avg_Rating":[4.2],

    "Complaints":[95],

    "Delivery_Time_Min":[30],

    "Service_Quality_Score":[82],

    "Complaint_Rate_Pct":[1.5],

    "Customer_Loyalty_Score":[63],

    "Repeat_Customer_Rate_Pct":[63]

})

In [ ]:
# Scenario 1 — Struggling Branch

In [22]:
high_risk_branch = pd.DataFrame({

    "Avg_Rating":[3.7],

    "Complaints":[240],

    "Delivery_Time_Min":[46],

    "Service_Quality_Score":[67],

    "Complaint_Rate_Pct":[3.8],

    "Customer_Loyalty_Score":[46],

    "Repeat_Customer_Rate_Pct":[46]

})

# Lets combine 3 scenarios

In [23]:
scenario_df = pd.concat([
    healthy_branch,
    medium_branch,
    high_risk_branch
])

scenario_df

,Avg_Rating,Complaints,Delivery_Time_Min,Service_Quality_Score,Complaint_Rate_Pct,Customer_Loyalty_Score,Repeat_Customer_Rate_Pct
0,4.8,20,18,95,0.3,82,82
0,4.2,95,30,82,1.5,63,63
0,3.7,240,46,67,3.8,46,46


# Lets Predict on new data 

In [24]:
predictions = model.predict(
    scenario_df
)

predictions

array([1, 2, 0])

# Convert Back To Labels

In [25]:
scenario_df["Predicted_Risk"] = (
    le.inverse_transform(predictions)
)


scenario_df

,Avg_Rating,Complaints,Delivery_Time_Min,Service_Quality_Score,Complaint_Rate_Pct,Customer_Loyalty_Score,Repeat_Customer_Rate_Pct,Predicted_Risk
0,4.8,20,18,95,0.3,82,82,Low
0,4.2,95,30,82,1.5,63,63,Medium
0,3.7,240,46,67,3.8,46,46,High


# Business Interpretation Layer

In [26]:
for index, row in scenario_df.iterrows():

    print("="*60)

    print(
        "Predicted Operational Risk:",
        row["Predicted_Risk"]
    )

    print("-"*60)

    print(
        f"Rating: {row['Avg_Rating']}"
    )

    print(
        f"Complaints: {row['Complaints']}"
    )

    print(
        f"Delivery Time: {row['Delivery_Time_Min']} mins"
    )

    print(
        f"Service Quality: {row['Service_Quality_Score']}"
    )

    print(
        f"Customer Loyalty: {row['Customer_Loyalty_Score']}"
    )

    print("="*60)

Predicted Operational Risk: Low
------------------------------------------------------------
Rating: 4.8
Complaints: 20
Delivery Time: 18 mins
Service Quality: 95
Customer Loyalty: 82
Predicted Operational Risk: Medium
------------------------------------------------------------
Rating: 4.2
Complaints: 95
Delivery Time: 30 mins
Service Quality: 82
Customer Loyalty: 63
Predicted Operational Risk: High
------------------------------------------------------------
Rating: 3.7
Complaints: 240
Delivery Time: 46 mins
Service Quality: 67
Customer Loyalty: 46


# Now Lets Build : Operational Risk Diagnosis Engine

# Step 1: Define feature columns

In [27]:
risk_features = [
    "Avg_Rating",
    "Complaints",
    "Delivery_Time_Min",
    "Service_Quality_Score",
    "Complaint_Rate_Pct",
    "Customer_Loyalty_Score",
    "Repeat_Customer_Rate_Pct"
]

# Step 2: Create dynamic branch diagnosis function

In [28]:
def diagnose_branch_operational_risk(branch_name, df, model, le):
    
    branch_df = df[df["Branch_Name"] == branch_name].sort_values("Month")
    
    if branch_df.empty:
        print("Branch not found. Please check branch name.")
        return
    
    latest_row = branch_df.tail(1)
    branch_features = latest_row[risk_features]
    
    prediction = model.predict(branch_features)[0]
    predicted_label = le.inverse_transform([prediction])[0]
    
    row = latest_row.iloc[0]
    
    print("="*80)
    print("OPERATIONAL RISK DIAGNOSIS REPORT")
    print("="*80)
    print("Branch:", branch_name)
    print("Month:", row["Month"])
    print("Predicted Operational Risk:", predicted_label)
    print("-"*80)
    
    print("Current Branch Metrics:")
    print(f"Average Rating: {row['Avg_Rating']}")
    print(f"Complaints: {row['Complaints']}")
    print(f"Delivery Time: {row['Delivery_Time_Min']} mins")
    print(f"Service Quality Score: {row['Service_Quality_Score']}")
    print(f"Customer Loyalty Score: {row['Customer_Loyalty_Score']}%")
    print(f"Repeat Customer Rate: {row['Repeat_Customer_Rate_Pct']}%")
    
    print("\nWhy this risk level was predicted:")
    
    reasons_found = False
    
    if row["Delivery_Time_Min"] > 35:
        print("- Delivery time is high. Customers may get frustrated because orders are taking too long.")
        reasons_found = True
        
    if row["Complaints"] > df["Complaints"].mean():
        print("- Complaints are above the business average. This shows customers are facing repeated issues.")
        reasons_found = True
        
    if row["Service_Quality_Score"] < df["Service_Quality_Score"].mean():
        print("- Service quality score is below average. This means customer experience needs improvement.")
        reasons_found = True
        
    if row["Avg_Rating"] < df["Avg_Rating"].mean():
        print("- Rating is below business average. This can reduce customer trust and online orders.")
        reasons_found = True
        
    if row["Customer_Loyalty_Score"] < df["Customer_Loyalty_Score"].mean():
        print("- Customer loyalty is below average. Fewer customers are returning to this branch.")
        reasons_found = True
    
    if not reasons_found:
        print("- Branch metrics are healthy compared to business average.")
    
    print("\nBusiness Impact:")
    
    if predicted_label == "High":
        print("- This branch needs immediate attention.")
        print("- If ignored, complaints may increase and repeat customers may reduce.")
        print("- Revenue may drop because customers may shift to competitors.")
        
    elif predicted_label == "Medium":
        print("- This branch is currently manageable but showing warning signs.")
        print("- If issues are not monitored, it may become high risk later.")
        
    else:
        print("- This branch is operating well.")
        print("- It can be used as a benchmark for weaker branches.")
    
    print("\nRecommended Actions:")
    
    if row["Delivery_Time_Min"] > 35:
        print("- Add delivery support during peak hours.")
        print("- Improve kitchen dispatch speed.")
        
    if row["Complaints"] > df["Complaints"].mean():
        print("- Review complaint reasons weekly.")
        print("- Improve packaging and order accuracy.")
        
    if row["Service_Quality_Score"] < df["Service_Quality_Score"].mean():
        print("- Train staff on service quality and order handling.")
        
    if row["Customer_Loyalty_Score"] < df["Customer_Loyalty_Score"].mean():
        print("- Launch repeat customer offers.")
        print("- Send recovery coupons to unhappy customers.")
        
    if predicted_label == "Low":
        print("- Continue current process.")
        print("- Use this branch as a benchmark.")
    
    print("\nExpected Outcome:")
    print("- Lower complaints")
    print("- Faster delivery")
    print("- Better ratings")
    print("- Higher repeat customers")
    print("- Reduced operational risk")
    print("="*80)

# Step 3: Test with real branch names

In [29]:
diagnose_branch_operational_risk(
    "Koramangala",
    df,
    model,
    le
)

OPERATIONAL RISK DIAGNOSIS REPORT
Branch: Koramangala
Month: 2025-12-01
Predicted Operational Risk: Low
--------------------------------------------------------------------------------
Current Branch Metrics:
Average Rating: 4.7
Complaints: 86
Delivery Time: 22 mins
Service Quality Score: 92.88
Customer Loyalty Score: 67.99567995679958%
Repeat Customer Rate: 68.0%

Why this risk level was predicted:
- Branch metrics are healthy compared to business average.

Business Impact:
- This branch is operating well.
- It can be used as a benchmark for weaker branches.

Recommended Actions:
- Continue current process.
- Use this branch as a benchmark.

Expected Outcome:
- Lower complaints
- Faster delivery
- Better ratings
- Higher repeat customers
- Reduced operational risk


In [30]:
diagnose_branch_operational_risk(
    "Electronic City",
    df,
    model,
    le
)

OPERATIONAL RISK DIAGNOSIS REPORT
Branch: Electronic City
Month: 2025-12-01
Predicted Operational Risk: High
--------------------------------------------------------------------------------
Current Branch Metrics:
Average Rating: 3.8
Complaints: 227
Delivery Time: 42 mins
Service Quality Score: 68.14
Customer Loyalty Score: 39.99079613437644%
Repeat Customer Rate: 39.99%

Why this risk level was predicted:
- Delivery time is high. Customers may get frustrated because orders are taking too long.
- Complaints are above the business average. This shows customers are facing repeated issues.
- Service quality score is below average. This means customer experience needs improvement.
- Rating is below business average. This can reduce customer trust and online orders.
- Customer loyalty is below average. Fewer customers are returning to this branch.

Business Impact:
- This branch needs immediate attention.
- If ignored, complaints may increase and repeat customers may reduce.
- Revenue may d

In [31]:
diagnose_branch_operational_risk(
    "Mahadevapura",
    df,
    model,
    le
)

OPERATIONAL RISK DIAGNOSIS REPORT
Branch: Mahadevapura
Month: 2025-12-01
Predicted Operational Risk: High
--------------------------------------------------------------------------------
Current Branch Metrics:
Average Rating: 4.0
Complaints: 185
Delivery Time: 46 mins
Service Quality Score: 74.3
Customer Loyalty Score: 41.98770491803279%
Repeat Customer Rate: 41.99%

Why this risk level was predicted:
- Delivery time is high. Customers may get frustrated because orders are taking too long.
- Complaints are above the business average. This shows customers are facing repeated issues.
- Service quality score is below average. This means customer experience needs improvement.
- Rating is below business average. This can reduce customer trust and online orders.
- Customer loyalty is below average. Fewer customers are returning to this branch.

Business Impact:
- This branch needs immediate attention.
- If ignored, complaints may increase and repeat customers may reduce.
- Revenue may drop 

In [32]:
df.to_csv(
    "Meghana_Feature_Engineered.csv",
    index=False
)